# Phase 5: Cross-stage and cross-language error analysis

Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 5: "where do the models fail (ORG spans? Bengali script
tokens? LLM format drift?)".

This notebook trains nothing and calls no LLM. It reloads the saved models, scores the full gold test sets, and compares
**which entities each stage gets wrong**.

| Question | Systems | Test set |
| --- | --- | --- |
| Where does each stage fail on Hindi? | BiLSTM-CRF, MuRIL, IndicBERT, LLM zero-shot and few-shot | 864 Hindi gold sentences |
| What changes between zero-shot and in-language models? | Hindi-trained vs in-language MuRIL, seeds 42, 43, 44 | Marathi and Bengali gold |
| Do Bengali script tokens explain the errors? | Same MuRIL models | Hindi, Marathi, Bengali gold |

**Safety rules**

- **`results/results.csv` is read only.** The notebook stops if its hash changes.
- **Every system must reproduce its logged F1 within 0.001** before any error is analyzed.
- LLM predictions are read from the committed Phase 5 run. The LLM is not run again.
- Outputs go to `results/error_analysis/`.

**How to run (Kaggle)**

1. Commit and push this notebook to `phase5-cross-stage-errors`, because the run clones that branch.
2. Import this notebook with **GPU T4 x2** and **Internet** on, and attach the `GH_TOKEN` Secret. The Phase 2
   `best.pt` is downloaded from its public Drive folder and verified by SHA-256.
3. With `DEBUG = True`, use **Save & Run All**. This scores 40 sentences per language. Keep nothing from it.
4. Set `DEBUG = False` and use **Save & Run All** again.
5. Download `NaamRekha/results/error_analysis/` and the executed notebook, unchanged.

In [ ]:
# Same pins as Phases 3 to 5, plus pytorch-crf from Phase 2.
%pip install -q transformers==5.22.1 accelerate==1.15.0 seqeval==1.2.2 sentencepiece==0.2.2 pytorch-crf==0.7.2

In [ ]:
import base64
import os
import subprocess
import sys

# Must be set before torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"
REPO_BRANCH = "phase5-cross-stage-errors"


def get_secret(name):
    # Kaggle Secret, then Colab Secret, then environment variable.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook. Check for a repo file, not
# just src/, because Kaggle has its own unrelated /kaggle/src folder.
if os.path.isfile("../src/data.py"):
    REPO_ROOT = os.path.abspath("..")
elif os.path.isfile("src/data.py"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("NaamRekha")
    if not os.path.isdir(REPO_ROOT):
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        token = get_secret("GH_TOKEN")
        if token:
            # Passed through the environment so it is never saved in .git/config
            # (Kaggle keeps that folder as output) or printed if the clone fails.
            auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            env.update(GIT_CONFIG_COUNT="1",
                       GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                       GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],
                       check=True, env=env)
sys.path.insert(0, REPO_ROOT)

hf_token = get_secret("HF_TOKEN")      # optional: every model used here is public
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

GIT_COMMIT = subprocess.check_output(["git", "-C", REPO_ROOT, "rev-parse", "--short", "HEAD"], text=True).strip()

from src import LABELS, evaluate, id2label, label2id, load_split

print("repo:", REPO_ROOT, "| commit:", GIT_COMMIT)
print("labels:", LABELS)

## Config

Model links and expected scores are read from `results/results.csv`, so they are not retyped here.

In [ ]:
DEBUG = True              # True: 40 sentences per language, nothing kept. False: the real run.
DEBUG_SENTENCES = 40

SOURCE_LANG = "hi"
TARGET_LANGS = ["mr", "bn"]
LANGS = [SOURCE_LANG, *TARGET_LANGS]
SEEDS = [42, 43, 44]
NATIVE_SCRIPT = {"hi": "Devanagari", "mr": "Devanagari", "bn": "Bengali"}

MAX_LENGTH = 128
EVAL_BATCH_SIZE = 32
REPRO_TOLERANCE = 0.001   # about two entities on the Hindi gold set; covers Phase 3's fp16 scoring
TOP_EXAMPLES = 12

# Phase 2 best.pt (epoch 4) from the public Drive folder linked in results.csv.
BILSTM_DRIVE_ID = "1r7RRTybH_j6Ir-3_SseX3iwoNmiGu-8H"
BILSTM_SHA256 = "ca058bc9ca4519c366c1d0c72ab0148c51e98540039f49ad04550190bc64f291"
BASE_TOKENIZER = "google/muril-base-cased"

# Phase 2 architecture.
WORD_DIM = 300
CHAR_DIM = 30
CHAR_HIDDEN = 25
HIDDEN = 128
DROPOUT = 0.5
MAX_WORD_CHARS = 20

In [ ]:
import csv
import gc
import glob
import gzip
import hashlib
import io
import json
import re
import tempfile
import unicodedata
from collections import Counter, defaultdict
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_colwidth", 70)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device, "| visible GPUs:", torch.cuda.device_count())
if not DEBUG and torch.cuda.device_count() != 1:
    raise RuntimeError(f"real runs need exactly one visible GPU, found {torch.cuda.device_count()}")

RESULTS_CSV = os.path.join(REPO_ROOT, "results", "results.csv")


def file_sha(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


RESULTS_SHA = file_sha(RESULTS_CSV)
OUT_DIR = tempfile.mkdtemp(prefix="naamrekha_errors_debug_") if DEBUG else os.path.join(REPO_ROOT, "results", "error_analysis")
os.makedirs(OUT_DIR, exist_ok=True)


def download_bilstm():
    path = os.path.join(tempfile.gettempdir(), "naamrekha_phase2_best.pt")
    if not os.path.isfile(path) or file_sha(path) != BILSTM_SHA256:
        url = f"https://drive.usercontent.google.com/download?id={BILSTM_DRIVE_ID}&confirm=t"
        subprocess.run(["curl", "-L", "--fail", "--silent", "--show-error", "--retry", "3", "-o", path, url], check=True)
    if file_sha(path) != BILSTM_SHA256:
        os.remove(path)
        raise RuntimeError("downloaded best.pt does not match the expected SHA-256")
    return path


BILSTM_FILE = download_bilstm()
print("BiLSTM-CRF checkpoint:", BILSTM_FILE, "| SHA-256 verified")
print("outputs ->", OUT_DIR)

## Data and run plan

Gold test sets are always the full shared splits, except in `DEBUG`.

In [ ]:
gold_all = {lang: load_split(lang, "test") for lang in LANGS}
gold = {lang: rows[:DEBUG_SENTENCES] if DEBUG else rows for lang, rows in gold_all.items()}
print({lang: len(rows) for lang, rows in gold.items()})

with open(RESULTS_CSV, encoding="utf-8", newline="") as f:
    SHEET = list(csv.DictReader(f))


def parse_notes(row):
    return dict(part.split("=", 1) for part in row["notes"].split("; ") if "=" in part)


def only(found, what):
    if len(found) != 1:
        raise RuntimeError(f"expected one results row for {what}, found {len(found)}")
    return found[0]


def phase4_row(train, test, seed):
    return only([r for r in SHEET if r["model"] == "MuRIL" and "phase=4" in r["notes"] and r["train_lang"] == train
                 and r["test_lang"] == test and parse_notes(r).get("seed") == str(seed)], f"MuRIL {train}->{test} s{seed}")


def pinned(row):
    m = re.fullmatch(r"https://huggingface\.co/([^/]+/[^/]+)/tree/([0-9a-f]{40})", row["checkpoint_path"])
    if not m:
        raise RuntimeError(f"not a pinned checkpoint: {row['checkpoint_path']}")
    return m.group(1), m.group(2)


META, REF, BERT_RUNS = {}, {}, []


def add_bert(sid, family, train_lang, seed, refs):
    repo, revision = pinned(next(iter(refs.values())))
    META[sid] = dict(family=family, train_lang=train_lang, seed=seed)
    for lang, row in refs.items():
        REF[(sid, lang)] = float(row["F1_overall"])
    BERT_RUNS.append(dict(sid=sid, repo=repo, revision=revision, langs=list(refs)))


for seed in SEEDS:
    add_bert(f"MuRIL-hi-s{seed}", "MuRIL", "hi", seed, {t: phase4_row("hi", t, seed) for t in LANGS})
    for lang in TARGET_LANGS:
        add_bert(f"MuRIL-{lang}-s{seed}", "MuRIL", lang, seed, {lang: phase4_row(lang, lang, seed)})

add_bert("IndicBERT", "IndicBERT", "hi", 42, {"hi": only([r for r in SHEET if r["model"] == "IndicBERT"], "IndicBERT")})

META["BiLSTM-CRF"] = dict(family="BiLSTM-CRF", train_lang="hi", seed=42)
REF[("BiLSTM-CRF", "hi")] = float(only([r for r in SHEET if r["model"] == "BiLSTM-CRF"], "BiLSTM-CRF")["F1_overall"])

LLM_SYSTEMS = {"LLM-zero": "LLM zero-shot [qwen2.5-7b-4bit]", "LLM-few": "LLM few-shot [qwen2.5-7b-4bit]"}
for sid, prefix in (("LLM-zero", "LLM-zero-shot ("), ("LLM-few", "LLM-few-shot (")):
    META[sid] = dict(family="LLM", train_lang="none", seed=42)
    REF[(sid, "hi")] = float(only([r for r in SHEET if r["model"].startswith(prefix)], prefix)["F1_overall"])

print(f"{len(BERT_RUNS)} transformer checkpoints | {len(REF)} system-language scores to reproduce")

## Prediction code

Transformer alignment and prediction are the same as Phase 4: labels are read from each word's first real subword, a
standalone `▁` is skipped, and truncated words count as `O`. Models are freshly loaded, so they run in full 32-bit.

In [ ]:
def label_positions(word_ids, tokens):
    # word -> position of the subword that carries its label: the first piece,
    # skipping a standalone SentencePiece "▁" when a real piece follows it.
    positions = {}
    for pos, wid in enumerate(word_ids):
        if wid is not None and (wid not in positions or tokens[positions[wid]] == "▁"):
            positions[wid] = pos
    return positions


def predict_tags(model, tokenizer, rows, max_length=MAX_LENGTH, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    dev = next(model.parameters()).device
    y_pred = []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        enc = tokenizer([r[0] for r in chunk], is_split_into_words=True, truncation=True,
                        max_length=max_length, padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            preds = model(**enc).logits.argmax(-1).cpu().numpy()
        for b, (_, tags) in enumerate(chunk):
            pred_tags = ["O"] * len(tags)
            for wid, pos in label_positions(enc.word_ids(batch_index=b), enc.tokens(batch_index=b)).items():
                pred_tags[wid] = id2label[int(preds[b][pos])]
            y_pred.append(pred_tags)
    return y_pred

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF


class BiLSTMCRF(nn.Module):
    # Same architecture and parameter names as notebooks/phase2-bilstm-crf.ipynb.
    def __init__(self, n_words, n_chars, n_tags):
        super().__init__()
        self.word_emb = nn.Embedding(n_words, WORD_DIM, padding_idx=0)
        self.char_emb = nn.Embedding(n_chars, CHAR_DIM, padding_idx=0)
        self.char_lstm = nn.LSTM(CHAR_DIM, CHAR_HIDDEN, batch_first=True, bidirectional=True)
        self.lstm = nn.LSTM(WORD_DIM + 2 * CHAR_HIDDEN, HIDDEN, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.proj = nn.Linear(2 * HIDDEN, n_tags)
        self.crf = CRF(n_tags, batch_first=True)

    def char_features(self, chars, char_lens):
        B, T, L = chars.shape
        flat, lens = chars.view(B * T, L), char_lens.view(-1)
        out = chars.new_zeros(B * T, 2 * CHAR_HIDDEN, dtype=torch.float)
        real = lens > 0
        packed = pack_padded_sequence(self.char_emb(flat[real]), lens[real].cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (h, _) = self.char_lstm(packed)
        out[real] = torch.cat([h[0], h[1]], dim=-1)
        return out.view(B, T, -1)

    def decode(self, words, chars, char_lens, mask):
        x = torch.cat([self.word_emb(words), self.char_features(chars, char_lens)], dim=-1)
        x = self.dropout(x)
        packed = pack_padded_sequence(x, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=words.size(1))
        return self.crf.decode(self.proj(self.dropout(h)), mask=mask)


def bilstm_batch(batch, stoi, ctoi):
    B, T = len(batch), max(len(w) for w in batch)
    encoded = [[[ctoi.get(ch, 1) for ch in x[:MAX_WORD_CHARS]] for x in w] for w in batch]
    L = max(len(cw) for sent in encoded for cw in sent)
    words = torch.zeros(B, T, dtype=torch.long)
    chars = torch.zeros(B, T, L, dtype=torch.long)
    char_lens = torch.zeros(B, T, dtype=torch.long)
    mask = torch.zeros(B, T, dtype=torch.bool)
    for b, (sent, char_sent) in enumerate(zip(batch, encoded)):
        words[b, :len(sent)] = torch.tensor([stoi.get(x, 1) for x in sent])
        mask[b, :len(sent)] = True
        for j, cw in enumerate(char_sent):
            chars[b, j, :len(cw)] = torch.tensor(cw)
            char_lens[b, j] = len(cw)
    return words, chars, char_lens, mask


def predict_bilstm(path, rows, dev, batch_size=256):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    if list(ck["labels"]) != LABELS:
        raise RuntimeError("BiLSTM-CRF checkpoint label order differs from the shared label map")
    stoi, ctoi = {w: i for i, w in enumerate(ck["itos"])}, {c: i for i, c in enumerate(ck["chars"])}
    model = BiLSTMCRF(len(ck["itos"]), len(ck["chars"]), len(LABELS))
    model.load_state_dict(ck["model"])
    model.to(dev).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(rows), batch_size):
            batch = [r[0] for r in rows[i:i + batch_size]]
            tensors = [t.to(dev) for t in bilstm_batch(batch, stoi, ctoi)]
            out.extend([id2label[x] for x in seq] for seq in model.decode(*tensors))
    del model
    return out

## Reload every system and reproduce its logged score

The run stops here if any score differs from `results.csv` by more than 0.001.

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

PREDS = {}


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


PREDS[("BiLSTM-CRF", "hi")] = predict_bilstm(BILSTM_FILE, gold["hi"], device)
free_memory()

for run in BERT_RUNS:
    tok = AutoTokenizer.from_pretrained(run["repo"], revision=run["revision"])
    model = AutoModelForTokenClassification.from_pretrained(run["repo"], revision=run["revision"]).to(device)
    if model.config.label2id != label2id:
        raise RuntimeError(f"{run['sid']}: checkpoint label map differs from src.label2id")
    for lang in run["langs"]:
        PREDS[(run["sid"], lang)] = predict_tags(model, tok, gold[lang])
    print("scored", run["sid"], run["langs"])
    del model, tok
    free_memory()

saved = defaultdict(dict)
with open(os.path.join(REPO_ROOT, "results", "phase5", "final_predictions.jsonl"), encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        if rec["system"] in LLM_SYSTEMS.values():
            saved[rec["system"]][rec["idx"]] = rec
for sid, name in LLM_SYSTEMS.items():
    recs = [saved[name][i] for i in range(len(gold_all["hi"]))]
    if any(r["words"] != w or r["gold"] != t for r, (w, t) in zip(recs, gold_all["hi"])):
        raise RuntimeError(f"{sid}: saved words or gold differ from the shared Hindi test split")
    PREDS[(sid, "hi")] = [r["pred"] for r in recs[:len(gold["hi"])]]

score_rows = []
for (sid, lang), pred in PREDS.items():
    s = evaluate([t for _, t in gold[lang]], pred)
    score_rows.append(dict(system=sid, lang=lang, **{k: round(v, 6) for k, v in s.items()}, logged=REF[(sid, lang)],
                           diff=s["F1_overall"] - REF[(sid, lang)]))
SCORES = pd.DataFrame(score_rows).sort_values(["lang", "system"]).reset_index(drop=True)
print(SCORES.round(4).to_string(index=False))
if len(PREDS) != len(REF):
    raise RuntimeError(f"scored {len(PREDS)} system-language pairs, expected {len(REF)}")
bad = SCORES[SCORES["diff"].abs() > REPRO_TOLERANCE]
if DEBUG:
    print("DEBUG: scores use a tiny slice, so reproduction is not checked.")
elif len(bad):
    raise RuntimeError("logged scores not reproduced:\n" + bad.to_string(index=False))
else:
    print(f"all {len(SCORES)} scores reproduce within {REPRO_TOLERANCE}")

## Error categories

These are the Phase 5 categories, so counts stay comparable:

- `missed`: no predicted span touches the gold entity.
- `boundary`: a predicted span overlaps it with different edges.
- `type_confusion`: exact span, wrong type.
- `spurious`: a predicted span touches no gold entity.

If one gold entity overlaps several predicted fragments, it still counts as one boundary error. `n_overlaps` records how
many fragments there were.

In [ ]:
from seqeval.metrics.sequence_labeling import get_entities

KINDS = ["boundary", "missed", "spurious", "type_confusion"]
TYPES = ["PER", "ORG", "LOC"]
SCRIPT_BLOCKS = {"Devanagari": (0x0900, 0x097F), "Bengali": (0x0980, 0x09FF)}


def spans(tags):
    return [(s, e, t) for t, s, e in get_entities(tags)]


def classify(words, gold_tags, pred_tags):
    G, P = spans(gold_tags), spans(pred_tags)
    gset, pset = set(G), set(P)
    text = lambda sp: " ".join(words[sp[0]:sp[1] + 1])
    out = []
    for g in G:
        if g in pset:
            continue
        same = [p for p in P if (p[0], p[1]) == (g[0], g[1])]
        over = [p for p in P if (p[0], p[1]) != (g[0], g[1]) and not (p[1] < g[0] or p[0] > g[1])]
        kind, p = ("type_confusion", same[0]) if same else ("boundary", over[0]) if over else ("missed", None)
        out.append(dict(kind=kind, gold=text(g), gold_type=g[2], pred=text(p) if p else "",
                        pred_type=p[2] if p else "", n_overlaps=len(over)))
    for p in P:
        if p not in gset and not any(not (g[1] < p[0] or g[0] > p[1]) for g in G):
            out.append(dict(kind="spurious", gold="", gold_type="", pred=text(p), pred_type=p[2], n_overlaps=0))
    return out


def char_script(ch):
    code_point = ord(ch)
    for name, (lo, hi) in SCRIPT_BLOCKS.items():
        if lo <= code_point <= hi:
            return name
    if ch.isascii() and ch.isalpha():
        return "Latin"
    return "Other"


def entity_script(text, lang):
    scripts = {char_script(ch) for ch in text if unicodedata.category(ch)[0] in ("L", "M")}
    if not scripts:
        return "no_letters"
    if scripts == {NATIVE_SCRIPT[lang]}:
        return "native"
    if scripts == {"Latin"}:
        return "latin"
    return "mixed_or_other"


def subword_bucket(n):
    return "1" if n <= 1 else "2" if n == 2 else "3-4" if n <= 4 else "5+"


def entity_table(lang, sids):
    # One row per gold entity, with one correct/incorrect column per system.
    records = []
    predicted = {sid: PREDS[(sid, lang)] for sid in sids}
    for i, (words, tags) in enumerate(gold[lang]):
        for s, e, t in spans(tags):
            rec = dict(lang=lang, idx=i, start=s, end=e, type=t, text=" ".join(words[s:e + 1]))
            records.append(rec)
        found = {sid: set(spans(predicted[sid][i])) for sid in sids}
        for rec in records[-len(spans(tags)):] if spans(tags) else []:
            key = (rec["start"], rec["end"], rec["type"])
            for sid in sids:
                rec[sid] = key in found[sid]
    return pd.DataFrame(records, columns=["lang", "idx", "start", "end", "type", "text", *sids])

In [ ]:
err_rows = []
for (sid, lang), pred in PREDS.items():
    for i, ((words, tags), p) in enumerate(zip(gold[lang], pred)):
        for e in classify(words, tags, p):
            err_rows.append(dict(system=sid, lang=lang, idx=i, **e))
ERR = pd.DataFrame(err_rows, columns=["system", "lang", "idx", "kind", "gold", "gold_type", "pred", "pred_type", "n_overlaps"])
ERR["type"] = ERR["gold_type"].where(ERR["gold_type"] != "", ERR["pred_type"])

COUNTS = ERR.groupby(["lang", "system", "kind"]).size().unstack(fill_value=0).reindex(columns=KINDS, fill_value=0)
COUNTS["total"] = COUNTS.sum(axis=1)
TYPE_COUNTS = ERR.groupby(["lang", "system", "type", "kind"]).size().unstack(fill_value=0).reindex(columns=KINDS, fill_value=0)
TYPE_COUNTS["total"] = TYPE_COUNTS.sum(axis=1)

# The same categories must reproduce the committed Phase 5 error table.
PHASE5_COUNTS = {"LLM-zero": (293, 299, 478, 52), "LLM-few": (317, 227, 406, 53), "MuRIL-hi-s42": (167, 77, 205, 31)}
if not DEBUG:
    for sid, expected in PHASE5_COUNTS.items():
        got = tuple(int(COUNTS.loc[("hi", sid), k]) for k in KINDS)
        if got != expected:
            raise RuntimeError(f"{sid}: Phase 5 error counts not reproduced: {got} vs {expected}")
    print("Phase 5 error counts reproduced for LLM zero-shot, LLM few-shot, and MuRIL")

## 1. Hindi across stages

All systems use the same 864 sentences. MuRIL is the seed 42 handoff model.

In [ ]:
HI_SYSTEMS = ["BiLSTM-CRF", "MuRIL-hi-s42", "IndicBERT", "LLM-zero", "LLM-few"]
print("Errors by kind:")
print(COUNTS.loc["hi"].reindex(HI_SYSTEMS).to_string())
print("\nErrors by type (gold type, or predicted type for spurious spans):")
print(TYPE_COUNTS.loc["hi"].reindex(pd.MultiIndex.from_product([HI_SYSTEMS, TYPES]), fill_value=0).to_string())

AGREE_SYSTEMS = ["BiLSTM-CRF", "MuRIL-hi-s42", "IndicBERT", "LLM-few"]
HI_ENT = entity_table("hi", AGREE_SYSTEMS)
HI_ENT["n_correct"] = HI_ENT[AGREE_SYSTEMS].sum(axis=1)
print(f"\nGold entities by number of systems correct (out of {len(AGREE_SYSTEMS)}):")
print(pd.crosstab(HI_ENT["type"], HI_ENT["n_correct"], margins=True).to_string())


def top_entities(frame, n=TOP_EXAMPLES):
    if frame.empty:
        return "none"
    c = frame.groupby(["type", "text"]).size().sort_values(ascending=False).head(n)
    return ", ".join(f"{t}:{x} ({k})" for (t, x), k in c.items())


print("\nWrong in every system:", top_entities(HI_ENT[HI_ENT["n_correct"] == 0]))
print("\nMuRIL right, BiLSTM-CRF wrong:", top_entities(HI_ENT[HI_ENT["MuRIL-hi-s42"] & ~HI_ENT["BiLSTM-CRF"]]))
print("\nBiLSTM-CRF right, MuRIL wrong:", top_entities(HI_ENT[HI_ENT["BiLSTM-CRF"] & ~HI_ENT["MuRIL-hi-s42"]]))
print("\nMuRIL right, IndicBERT wrong:", top_entities(HI_ENT[HI_ENT["MuRIL-hi-s42"] & ~HI_ENT["IndicBERT"]]))
print("\nMuRIL right, LLM few-shot wrong:", top_entities(HI_ENT[HI_ENT["MuRIL-hi-s42"] & ~HI_ENT["LLM-few"]]))
print("\nLLM few-shot right, MuRIL wrong:", top_entities(HI_ENT[HI_ENT["LLM-few"] & ~HI_ENT["MuRIL-hi-s42"]]))

## 2. Zero-shot vs in-language on Marathi and Bengali

Differences are paired by seed: `in-language errors - zero-shot errors`. A **negative** value means the in-language model
makes fewer errors. `same_sign` shows whether all three seeds agree. Seeds measure training variation, not test-set
sampling variation.

"Consistent wins" are gold entities that all three models on one side get right and all three on the other side get wrong.

In [ ]:
def error_count(sid, lang, kind=None, etype=None):
    if etype is None:
        if (lang, sid) not in COUNTS.index:      # a tiny DEBUG slice can have no errors
            return 0
        return int(COUNTS.loc[(lang, sid), "total" if kind is None else kind])
    key = (lang, sid, etype)
    if key not in TYPE_COUNTS.index:
        return 0
    return int(TYPE_COUNTS.loc[key, "total" if kind is None else kind])


gap_rows = []
for lang in TARGET_LANGS:
    for etype in [None, *TYPES]:
        for kind in [None, *KINDS]:
            zero = [error_count(f"MuRIL-hi-s{s}", lang, kind, etype) for s in SEEDS]
            own = [error_count(f"MuRIL-{lang}-s{s}", lang, kind, etype) for s in SEEDS]
            delta = [b - a for a, b in zip(zero, own)]
            gap_rows.append(dict(lang=lang, type=etype or "ALL", kind=kind or "total", zero_shot_mean=np.mean(zero),
                                 in_language_mean=np.mean(own), diff_mean=np.mean(delta),
                                 diff_sd=np.std(delta, ddof=1) if not DEBUG or len(delta) > 1 else 0.0,
                                 per_seed="/".join(f"{d:+d}" for d in delta),
                                 same_sign=all(d > 0 for d in delta) or all(d < 0 for d in delta)))
TRANSFER_GAPS = pd.DataFrame(gap_rows)
print(TRANSFER_GAPS.round(2).to_string(index=False))

win_rows, TRANSFER_ENT = [], {}
for lang in TARGET_LANGS:
    zero_sids = [f"MuRIL-hi-s{s}" for s in SEEDS]
    own_sids = [f"MuRIL-{lang}-s{s}" for s in SEEDS]
    ent = entity_table(lang, zero_sids + own_sids)
    TRANSFER_ENT[lang] = ent
    zero_wins = ent[ent[zero_sids].all(axis=1) & ~ent[own_sids].any(axis=1)]
    own_wins = ent[ent[own_sids].all(axis=1) & ~ent[zero_sids].any(axis=1)]
    for etype in [*TYPES, "ALL"]:
        z = zero_wins if etype == "ALL" else zero_wins[zero_wins["type"] == etype]
        o = own_wins if etype == "ALL" else own_wins[own_wins["type"] == etype]
        total = len(ent) if etype == "ALL" else int((ent["type"] == etype).sum())
        win_rows.append(dict(lang=lang, type=etype, gold_entities=total, zero_shot_wins=len(z), in_language_wins=len(o)))
    print(f"\n{lang}: Hindi models consistently right, in-language consistently wrong:", top_entities(zero_wins))
    print(f"{lang}: in-language consistently right, Hindi models consistently wrong:", top_entities(own_wins))
CONSISTENT_WINS = pd.DataFrame(win_rows)
print("\nConsistent wins:")
print(CONSISTENT_WINS.to_string(index=False))

## 3. Script and tokenizer effects

This section tests the roadmap's "Bengali script tokens?" question in three ways: the script of each gold entity,
`[UNK]` subwords, and how many MuRIL subwords an entity needs. Recall is averaged across the three seeds.

Different languages use different test sets, so compare patterns, not absolute values across languages.

In [ ]:
base_tok = AutoTokenizer.from_pretrained(BASE_TOKENIZER)
word_cache = {}


def word_info(word):
    if word not in word_cache:
        ids = base_tok(word, add_special_tokens=False)["input_ids"]
        word_cache[word] = (len(ids), base_tok.unk_token_id in ids)
    return word_cache[word]


token_rows = []
for lang in LANGS:
    info = [word_info(w) for words, _ in gold[lang] for w in words]
    token_rows.append(dict(lang=lang, words=len(info), mean_subwords=np.mean([n for n, _ in info]),
                           pct_words_multi_subword=100 * np.mean([n > 1 for n, _ in info]),
                           pct_words_with_unk=100 * np.mean([u for _, u in info])))
TOKEN_STATS = pd.DataFrame(token_rows)
print(TOKEN_STATS.round(3).to_string(index=False))

SCRIPT_ENT, recall_rows = {}, []
for lang in LANGS:
    zero_sids = [f"MuRIL-hi-s{s}" for s in SEEDS]
    own_sids = [] if lang == "hi" else [f"MuRIL-{lang}-s{s}" for s in SEEDS]
    ent = entity_table(lang, zero_sids + own_sids).copy()
    infos = [[word_info(w) for w in gold[lang][r.idx][0][r.start:r.end + 1]] for r in ent.itertuples()]
    ent["script"] = [entity_script(t, lang) for t in ent["text"]]
    ent["subwords"] = [subword_bucket(sum(n for n, _ in x)) for x in infos]
    ent["has_unk"] = ["unk" if any(u for _, u in x) else "no_unk" for x in infos]
    SCRIPT_ENT[lang] = ent
    groups = {"hindi_model": zero_sids}
    if own_sids:
        groups["in_language"] = own_sids
    for factor in ("script", "subwords", "has_unk"):
        for level, frame in ent.groupby(factor):
            row = dict(lang=lang, factor=factor, level=level, entities=len(frame))
            for name, sids in groups.items():
                row[f"{name}_recall"] = frame[sids].to_numpy().mean() if len(frame) else np.nan
            recall_rows.append(row)
SCRIPT_RECALL = pd.DataFrame(recall_rows)
print(SCRIPT_RECALL.round(4).to_string(index=False))

## Save outputs

Predictions and error rows are compressed, and the summary tables are plain CSV. `results.csv` must still be unchanged.

In [ ]:
def write_gzip_text(path, writer):
    with open(path, "wb") as raw, gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as gz, \
            io.TextIOWrapper(gz, encoding="utf-8", newline="\n") as text:
        writer(text)


def write_predictions(text):
    for (sid, lang), pred in sorted(PREDS.items()):
        for i, tags in enumerate(pred):
            text.write(json.dumps(dict(system=sid, lang=lang, idx=i, pred="".join(str(label2id[t]) for t in tags)),
                                  ensure_ascii=False, separators=(",", ":")) + "\n")


write_gzip_text(os.path.join(OUT_DIR, "predictions.jsonl.gz"), write_predictions)
write_gzip_text(os.path.join(OUT_DIR, "errors.csv.gz"), lambda text: ERR.to_csv(text, index=False))
for name, frame, index in (("scores", SCORES, False), ("error_counts", COUNTS.reset_index(), False),
                           ("error_counts_by_type", TYPE_COUNTS.reset_index(), False),
                           ("transfer_gaps", TRANSFER_GAPS, False), ("consistent_wins", CONSISTENT_WINS, False),
                           ("token_stats", TOKEN_STATS, False), ("script_recall", SCRIPT_RECALL, False)):
    frame.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=index, encoding="utf-8", lineterminator="\n")

if file_sha(RESULTS_CSV) != RESULTS_SHA:
    raise RuntimeError("results/results.csv changed during this run")

run_info = dict(git=GIT_COMMIT, debug=DEBUG, results_csv_sha256=RESULTS_SHA, n_systems=len(META), n_scores=len(SCORES),
                sentences={lang: len(rows) for lang, rows in gold.items()},
                bilstm_drive_id=BILSTM_DRIVE_ID, bilstm_sha256=BILSTM_SHA256,
                versions={p: version(p) for p in ("transformers", "accelerate", "seqeval", "sentencepiece", "pytorch-crf")},
                torch=torch.__version__, label_map=LABELS)
with open(os.path.join(OUT_DIR, "run_info.json"), "w", encoding="utf-8", newline="\n") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=1)
print("saved:", sorted(os.listdir(OUT_DIR)))
print("results.csv unchanged")
print("cross-stage error analysis complete | debug:", DEBUG)

## Findings

Written after the real run, from the printed tables above. The write-up belongs in `docs/phase5/error_analysis.md` and
Sections 7 and 8 of `docs/phase6/report.md`.